# Loading Gathered data from 10-k files of Apple, Microsoft, Tesla
## _Total Revenue, Net Income, Total Assests, Total liabilities, Cach flow from Operating Activities_

In [2]:
import pandas as pd

In [3]:
# Loading data
df = pd.read_csv("10-k reports for Testa, Microsoft, Apple.csv")

df['Net Income'] = df['Net Income'] / 1000000
df['Total Revenue'] = df['Total Revenue'] / 1000000
df['Total Assets'] = df['Total Assets'] / 1000000
df['Total Liabilities'] = df['Total Liabilities'] / 1000000
df['Cash Flow from Operating Activities'] = df['Cash Flow from Operating Activities'] / 1000000

# change Cash Flow from Operating Activities to Operating cach flow
# df.rename(columns={'Cash Flow from Operating Activities': 'Operating cash flow'}, inplace=True)


print(df)

  Company name  10-k Year  Total Revenue  Net Income  Total Assets  \
0        Apple       2025       416161.0    112010.0      359241.0   
1        Apple       2024       391035.0     93736.0      364980.0   
2        Apple       2023       383285.0     96995.0      352583.0   
3    Microsoft       2025       281724.0    101832.0      619003.0   
4    Microsoft       2024       245122.0     88136.0      512163.0   
5    Microsoft       2023       211915.0     72361.0      411976.0   
6        Tesla       2025        94827.0      3855.0      137806.0   
7        Tesla       2024        97690.0      7153.0      122070.0   
8        Tesla       2023        96773.0     14974.0       82338.0   

   Total Liabilities  Cash Flow from Operating Activities  
0           285508.0                             111482.0  
1           308030.0                             118254.0  
2           290437.0                             110543.0  
3           275524.0                             136162.0  

In [10]:
test = df[df["Company name"].str.lower() == "apple"].sort_values("10-k Year")

print(test)

  Company name  10-k Year  Total Revenue  Net Income  Total Assets  \
2        Apple       2023       383285.0     96995.0      352583.0   
1        Apple       2024       391035.0     93736.0      364980.0   
0        Apple       2025       416161.0    112010.0      359241.0   

   Total Liabilities  Cash Flow from Operating Activities  
2           290437.0                             110543.0  
1           308030.0                             118254.0  
0           285508.0                             111482.0  


In [8]:
df = df.sort_values(['Company name', '10-k Year'], ascending=[True, True])

df['Revenue Growth (%)'] = df.groupby('Company name')['Total Revenue'].pct_change() * 100
df['Net Income Growth (%)'] = df.groupby('Company name')['Net Income'].pct_change() * 100
df['Net Margin (%)']     = df['Net Income'] / df['Total Revenue'] * 100
df['Debt-to-Assets (%)'] = df['Total Liabilities'] / df['Total Assets'] * 100

In [9]:
print(df)

  Company name  10-k Year  Total Revenue  Net Income  Total Assets  \
2        Apple       2023       383285.0     96995.0      352583.0   
1        Apple       2024       391035.0     93736.0      364980.0   
0        Apple       2025       416161.0    112010.0      359241.0   
5    Microsoft       2023       211915.0     72361.0      411976.0   
4    Microsoft       2024       245122.0     88136.0      512163.0   
3    Microsoft       2025       281724.0    101832.0      619003.0   
8        Tesla       2023        96773.0     14974.0       82338.0   
7        Tesla       2024        97690.0      7153.0      122070.0   
6        Tesla       2025        94827.0      3855.0      137806.0   

   Total Liabilities  Cash Flow from Operating Activities  Revenue Growth (%)  \
2           290437.0                             110543.0                 NaN   
1           308030.0                             118254.0            2.021994   
0           285508.0                             111482.

In [10]:
df.to_csv('10k_analysis.csv', index=False, float_format='%.2f')

# 10-K Financial Analysis
**Companies:** Apple · Microsoft · Tesla
**Period:** FY2023 – FY2025
**Data Source:** Annual 10-K filings (SEC EDGAR)

---

## 1. Methodology

### Data Collection
Financial figures were extracted manually from each company's 10-K filings for three fiscal years. All values are reported in **millions of USD ($M)**.

The five raw metrics collected per company per year:

| Metric | Description |
|---|---|
| Total Revenue | All income generated from operations |
| Net Income | Profit after all expenses and taxes |
| Total Assets | Everything the company owns |
| Total Liabilities | Everything the company owes |
| Operating Cash Flow | Cash generated from core business operations |

### Data Processing (pandas)
The data was loaded into a pandas DataFrame and sorted **ascending by year** (2023 → 2025) within each company group. This ordering is critical — `pct_change()` compares row N to row N-1, so the data must flow forward in time or the growth rates come out inverted.

```python
df = df.sort_values(['Company', '10-K Year']).reset_index(drop=True)
```

### Calculated Metrics

**Year-over-Year (YoY) Growth** — computed with `groupby` + `pct_change()` so each company is calculated independently:

```python
df['Revenue YoY (%)']    = df.groupby('Company')['Total Revenue ($M)'].pct_change() * 100
df['Net Income YoY (%)'] = df.groupby('Company')['Net Income ($M)'].pct_change() * 100
```

> The 2023 row for each company shows `NaN` for growth metrics — this is correct and expected, as 2023 is the base year with no prior year to compare against.

**Profitability & Leverage Ratios** — derived directly from the raw figures:

```python
df['Net Margin (%)']     = df['Net Income ($M)'] / df['Total Revenue ($M)'] * 100
df['Debt-to-Assets (%)'] = df['Total Liabilities ($M)'] / df['Total Assets ($M)'] * 100
```

---

## 2. Results

### Full Dataset

| Company | Year | Revenue ($M) | Net Income ($M) | Total Assets ($M) | Total Liab. ($M) | Op. Cash Flow ($M) | Revenue YoY % | Net Income YoY % | Net Margin % | Debt-to-Assets % |
|---|---|---|---|---|---|---|---|---|---|---|
| Apple | 2023 | 383,285 | 96,995 | 352,583 | 290,437 | 110,543 | — | — | 25.31% | 82.37% |
| Apple | 2024 | 391,035 | 93,736 | 364,980 | 308,030 | 118,254 | +2.02% | -3.36% | 23.97% | 84.40% |
| Apple | 2025 | 416,161 | 112,010 | 359,241 | 285,508 | 111,482 | +6.43% | +19.50% | 26.92% | 79.48% |
| Microsoft | 2023 | 211,915 | 72,361 | 411,976 | 205,753 | 87,582 | — | — | 34.15% | 49.94% |
| Microsoft | 2024 | 245,122 | 88,136 | 512,163 | 243,686 | 118,548 | +15.67% | +21.80% | 35.96% | 47.58% |
| Microsoft | 2025 | 281,724 | 101,832 | 619,003 | 275,524 | 136,162 | +14.93% | +15.54% | 36.15% | 44.51% |
| Tesla | 2023 | 96,773 | 14,974 | 82,338 | 36,440 | 13,256 | — | — | 15.47% | 44.26% |
| Tesla | 2024 | 97,690 | 7,153 | 122,070 | 48,390 | 14,923 | +0.95% | -52.23% | 7.32% | 39.64% |
| Tesla | 2025 | 94,827 | 3,855 | 137,806 | 54,941 | 14,747 | -2.93% | -46.11% | 4.07% | 39.87% |

---

## 3. Observations

### Apple
- Revenue grew steadily from $383B to $416B, accelerating to **+6.4% in FY2025**.
- Net income dipped slightly in 2024 (-3.4%) but rebounded strongly in 2025 (+19.5%), pushing the net margin back up to **26.9%** — a healthy sign of operating leverage.
- Debt-to-assets is high (~80%) but improved in 2025, consistent with Apple's well-known strategy of carrying significant liabilities while returning cash to shareholders via buybacks and dividends.
- Operating cash flow remained remarkably stable (~$110–118B), showing the underlying business is consistently generating cash regardless of year-to-year income swings.

### Microsoft
- The strongest and most consistent performer of the three. Revenue grew **+15.7% in 2024** and **+14.9% in 2025**, reflecting continued cloud (Azure) and AI-driven expansion.
- Net margin is the highest of the group and nearly flat across all three years (~34–36%), indicating disciplined cost control as the business scales.
- Debt-to-assets is moderate and **declining** (50% → 45%), meaning assets are growing faster than liabilities — a sign of a strengthening balance sheet.
- Operating cash flow grew from $87B to $136B over the period, a **+55% increase in two years**, which is exceptional at this scale.

### Tesla
- The most concerning trajectory of the three. Revenue was essentially **flat from 2023 to 2024** (+0.95%) and **declined in 2025** (-2.93%), while assets grew significantly — meaning the company is investing heavily without generating more revenue.
- Net income collapsed from $14.9B in 2023 to $3.9B in 2025, a **-74% drop over two years**, driven by aggressive price cuts and margin compression.
- Net margin fell from 15.5% to just 4.1%, which is a dramatic deterioration in profitability.
- Debt-to-assets is the lowest of the three (~40%) and stable, so leverage is not the issue — the core challenge is profitability under pricing pressure.

---

## 4. Conclusions

| | Apple | Microsoft | Tesla |
|---|---|---|---|
| Revenue Trend | ↑ Growing | ↑↑ Fast growth | ↓ Declining |
| Profitability Trend | → Stable/recovering | ↑ Expanding | ↓↓ Deteriorating |
| Balance Sheet | High leverage, stable | Strengthening | Asset-heavy, low profit |
| Cash Generation | Strong & consistent | Rapidly growing | Modest |
| Overall Signal | 🟡 Steady | 🟢 Strong | 🔴 Under pressure |

**Microsoft** stands out as the strongest across all metrics — consistent revenue growth, expanding margins, a strengthening balance sheet, and rapidly growing cash flow. This reflects its successful pivot to cloud and AI services.

**Apple** is a mature, cash-generating machine. Revenue and profit are growing, and despite high headline leverage, the underlying cash flow (~$111B/year) provides enormous financial flexibility.

**Tesla** is the outlier with a deteriorating financial profile. The combination of flat/declining revenue and collapsing net income is a warning sign. The key question going forward is whether Tesla can stabilize margins or find new revenue streams (energy, autonomy) to offset pressure on its core auto business.

---

*Analysis conducted using Python (pandas). Raw data sourced from SEC 10-K filings.*

In [44]:
summary = df.groupby('Company name').agg({
    'Revenue Growth (%)': 'mean',
    'Net Income Growth (%)': 'mean'
}).reset_index()

print(summary)

  Company name  Revenue Growth (%)  Net Income Growth (%)
0        Apple            4.223753               8.067605
1    Microsoft           15.301059              18.670019
2        Tesla           -0.991560             -49.168531
